# P02 — GEARS training contract (Phase 3 engineering layer)

**This notebook documents the per-seed GEARS training contract.** Full GEARS matched-n retraining is documented in `TRAINING_PROVENANCE.md` (repository root) but is not the default reproduction path because it requires model-specific dependencies and long-running stochastic training. The manuscript's 28 GEARS prediction h5ads (matched-n cross-architecture audit, 4 conditions × 7 seeds = 28 runs, reported in Results §3.5 and Table 2) were generated by author scripts described in that document; this notebook records the inputs, hyperparameters, and outputs that any replacement implementation would need to satisfy, and exits without performing work in the default state.

For manuscript reproduction, use the default public path:

```bash
./run_all.sh --figures
```

which reproduces every manuscript figure, table, and Appendix B (including the matched-n GEARS Table 2) from the shipped evaluation CSVs in `precomputed/` in ~2 minutes, without atlases, without `gears`, and without training.

**Manuscript reference:** Methods §2.2 (closing paragraph on the GEARS audit); Results §3.5.

**If you want to retrain anyway:** both `QUICK_DEMO=1` and `FULL_TRAINING=1` run the same 28 GEARS jobs (4 conditions × 7 seeds), because the matched-n audit is small enough that a partial run is not informative. Both require the substantive training loop in `src/perturb_eval/train_gears.py`, which is currently a contract-only module.

**Inputs (if executed):**
- `data/replogle/{k562,rpe1}_essential.h5ad` (from D01).
- `data/holdout_specs/holdout_specs.csv` (from D03; only seeds 1–7 of each condition are used).
- `data/severity_refs/replogle_{K562,RPE1}_severity.csv` (from D02).

**Outputs (one per condition × seed):**
- `data/predictions/severity_details/GEARS_{cell_type}_{split_type}_seed{seed}.severity.h5ad`

**Dependencies (if executed):** requires the `gears` dependency group:
```bash
uv sync --group gears
```


In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from perturb_style import DATA_DIR

QUICK_DEMO = os.environ.get("QUICK_DEMO", "0") == "1"
FULL_TRAINING = os.environ.get("FULL_TRAINING", "0") == "1"

if QUICK_DEMO or FULL_TRAINING:
    SEEDS = list(range(1, 8))   # matched-n: seeds 1-7
    mode = "QUICK_DEMO" if QUICK_DEMO else "FULL_TRAINING"
    print(f"Mode: {mode}  (7 seeds per condition; 28 runs total; ~2 hours on GPU/MPS)")
else:
    SEEDS = []
    print("No training mode set. P02 performs no work without QUICK_DEMO=1 or FULL_TRAINING=1.")
    print()
    print("If you only need to reproduce the figures and tables, run:")
    print("  ./run_all.sh --figures")


No training mode set. P02 performs no work without QUICK_DEMO=1 or FULL_TRAINING=1.

If you only need to reproduce the figures and tables, run:
  ./run_all.sh --figures


## Check prerequisites

In [2]:
ATLAS_PATHS = {
    "K562": DATA_DIR / "replogle" / "k562_essential.h5ad",
    "RPE1": DATA_DIR / "replogle" / "rpe1_essential.h5ad",
}
SPEC_PATH = DATA_DIR / "holdout_specs" / "holdout_specs.csv"
SEV_REFS = {
    "K562": DATA_DIR / "severity_refs" / "replogle_K562_severity.csv",
    "RPE1": DATA_DIR / "severity_refs" / "replogle_RPE1_severity.csv",
}
OUT_DIR = DATA_DIR / "predictions" / "severity_details"
OUT_DIR.mkdir(exist_ok=True, parents=True)

prerequisites_ok = SEEDS and all(p.exists() for p in [*ATLAS_PATHS.values(), SPEC_PATH, *SEV_REFS.values()])

if SEEDS and not prerequisites_ok:
    print("Training mode set but prerequisites are missing:")
    for name, path in [
        ("K562 atlas", ATLAS_PATHS["K562"]),
        ("RPE1 atlas", ATLAS_PATHS["RPE1"]),
        ("Holdout specs", SPEC_PATH),
        ("K562 severity", SEV_REFS["K562"]),
        ("RPE1 severity", SEV_REFS["RPE1"]),
    ]:
        marker = "OK" if path.exists() else "MISSING"
        print(f"  [{marker}] {name}: {path}")
    print()
    print("Run D01, D02, D03 first.")
elif SEEDS:
    print("All prerequisites present.")


## Train GEARS per condition × seed (matched-n only)

**Layer status.** As with P01, full retraining is a Phase 3 engineering layer that is deliberately not packaged as the default public reproducibility path; see `TRAINING_PROVENANCE.md` for the rationale and for the author scripts that produced the 28 GEARS prediction h5ads shipped with this repository (matched-n cross-architecture audit, 4 conditions × 7 seeds = 28 runs). The cell below records the training contract that any replacement implementation must satisfy, and links to `src/perturb_eval/train_gears.py` for the orchestration entry point. The default public reproducibility path (`./run_all.sh --figures`) bypasses training entirely and reproduces the manuscript Table 2 matched-n audit from shipped evaluation artefacts.

In [3]:
if SEEDS and prerequisites_ok:
    from itertools import product

    CONDITIONS = [(c, s) for c in ("K562", "RPE1") for s in ("random", "stratified")]
    total = len(CONDITIONS) * len(SEEDS)
    print(f"Plan: {total} GEARS training runs ({len(CONDITIONS)} conditions x {len(SEEDS)} seeds).")
    print()

    try:
        from perturb_eval.train_gears import train_one_seed
    except ImportError:
        train_one_seed = None

    if train_one_seed is None:
        print("Training is deferred to the Phase 3 engineering layer.")
        print("See TRAINING_PROVENANCE.md for the author scripts that produced the")
        print("manuscript's 28 GEARS prediction h5ads (matched-n cross-architecture")
        print("audit, 4 conditions x 7 seeds), and for the validation chain")
        print("(raw predictions -> 28 severity h5ads -> Table 2 + Discussion §4.2).")
        print()
        print("This notebook is the public-repo orchestration contract: any replacement")
        print("implementation of perturb_eval.train_gears.train_one_seed should satisfy")
        print("the following per-seed flow.")
        print()
        print("Per-seed contract:")
        print("  1. Read atlas h5ad for cell_type.")
        print("  2. Look up holdout perturbations for (cell_type, split_type, seed) from D03 spec.")
        print("  3. Build the GEARS PertData object from atlas minus holdout.")
        print("  4. Train GEARS with seed=seed for 30 epochs (or GEARS default).")
        print("  5. Predict per-cell expression on the 30-perturbation holdout.")
        print("  6. Aggregate to per-perturbation predicted_mean_abs_delta.")
        print("  7. Attach leverage_score column from D02 severity reference.")
        print("  8. Write data/predictions/severity_details/GEARS_{cell}_{split}_seed{N}.severity.h5ad")
        print("     with .obs columns predicted_mean_abs_delta, leverage_score, perturbation_target.")
        print()
        print("To validate that any new prediction satisfies the eval-layer contract:")
        print("  uv run python scripts/evaluate_severity_panel.py --validate-existing \\")
        print("    data/predictions/severity_details")
    else:
        ok, missing = 0, []
        for (cell, split), seed in product(CONDITIONS, SEEDS):
            try:
                path = train_one_seed(cell_type=cell, split_type=split, seed=seed,
                                       atlas_path=ATLAS_PATHS[cell],
                                       spec_path=SPEC_PATH,
                                       severity_ref_path=SEV_REFS[cell],
                                       output_dir=OUT_DIR)
                ok += 1
                print(f"  [{ok}/{total}] GEARS {cell} {split} seed{seed} -> {path.name}")
            except Exception as e:
                missing.append((cell, split, seed, str(e)))
                print(f"  FAIL GEARS {cell} {split} seed{seed}: {e}")

        print(f"\nCompleted: {ok}/{total} runs successful; {len(missing)} failed.")
